In [ ]:
import numpy as np 
from pprint import pprint
import duckdb

from plant_pheno.review import review_label_issues
from plant_pheno.config import CLASS_ORDER

In [151]:
pprint(obs_ids[:3])
pprint(raw_labels[:3])
pprint(raw_preds[:3])

[1848427, 6157586, 6327803]
array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])
array([[0.01098633, 0.02099609, 0.01245117],
       [0.01281738, 0.11572266, 0.01281738],
       [0.01361084, 0.01696777, 0.01745605]])


In [152]:
con = duckdb.connect('/home/etienne/projects/inat-phenology-cv/data/cleanlab.duckdb')
result = con.execute("""
    SELECT obs_id, labels, preds
    FROM test_set
    ORDER BY obs_id
""").fetchall()
obs_ids = [row[0] for row in result]
raw_labels = np.array([np.array(row[1]) for row in result])
raw_preds = np.array([np.array(row[2]) for row in result])
con.close()

In [153]:
pprint(obs_ids[:3])
pprint(raw_labels[:3])
pprint(raw_preds[:3])

[1848427, 6157586, 6327803]
array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])
array([[0.01098633, 0.02099609, 0.01245117],
       [0.01281738, 0.11572266, 0.01281738],
       [0.01361084, 0.01696777, 0.01745605]])


In [154]:
from cleanlab.filter import find_label_issues
import numpy as np

means = []
issues = []

for i in range(3):
    # Format predicted probs for cleanlab
    labels = raw_labels[:,i].astype(int)
    pred_probs_pos = raw_preds[:,i]
    pred_probs_neg = 1 - pred_probs_pos
    pred_probs = np.column_stack((pred_probs_neg, pred_probs_pos))
    issue_mask = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    )
    means.append(issue_mask.mean())

    ordered_issue_indices = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence"
    )
    
    issues.append([obs_ids[i] for i in ordered_issue_indices])

In [155]:
for i, _ in enumerate(CLASS_ORDER):
    print(f"{CLASS_ORDER[i]} {means[i]}")
    print(f" {len(issues[i])} issues")

Flowering 0.02023608768971332
 36 issues
Fruiting 0.07307476110174255
 130 issues
Flower_Budding 0.15177065767284992
 270 issues


In [ ]:
class_idx = 1

x = review_label_issues(obs_ids= issues[class_idx],
 label_name= CLASS_ORDER[class_idx],
 db_path= "/home/etienne/projects/inat-phenology-cv/data/cv_raw.duckdb",
 image_dir= "/home/etienne/projects/inat-phenology-cv/data/images",
 table_name='cv_photos3',
 all_obs_ids=obs_ids,
 raw_labels=raw_labels,
 raw_preds=raw_preds,
 class_idx=class_idx
 )

In [157]:
ids = []

for i in x:
    print(i['obs_id'])


89018160
167635796
47229890
116918737
226014764
25229715
80188100
83400158
120029911
132889106
56536637
307068898
220985985
233484681
194383072
195658719
302233775
231102396
15379803
323106146
152253395
54121865
102472867
211729106
77062792
289295279
126275900
144601745
183201382
74387516
224117475
307068743
104974932
284701213
216705335
78418280
